# 🌦️ A Predictive Micro-Climate Platform for Kenyan Smallholder Farmers
## 🚀 Cloud Machine Learning Training Pipeline (Google Colab GPU Accelerator)
**Strathmore University Capstone Project 2026**  
**Author:** Munyirwa Joshua Kitto | **Supervisor:** Daniel Machanje  

---

### 📌 How to use this notebook:
1. **Enable Free T4 GPU:** In the top menu, go to **Runtime** → **Change runtime type** → select **T4 GPU** under Hardware accelerator, then click **Save**.
2. **Run Cells Sequentially:** Press `Shift + Enter` on each cell from top to bottom.
3. **Download Weights:** The final cell will automatically zip and download `trained_weights.zip` to your computer.
4. **Deploy Locally:** Unzip `trained_weights.zip` into your local `hyper-local weather/ml_engine/weights/` folder.

### 1. Verify GPU Availability
Verify that Google Colab has allocated an NVIDIA GPU (e.g. Tesla T4).

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: GPU is not active. Go to Runtime -> Change runtime type -> T4 GPU.")

### 2. Install Python Dependencies
Colab includes PyTorch, Scikit-Learn, and Pandas by default. We install XGBoost and required utilities.

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy python-dotenv joblib matplotlib seaborn

### 3. Load Project Files & Execute GPU Training Pipeline
If `colab_upload.zip` is not already in your Colab files, a **'Choose Files'** button will prompt you to select `colab_upload.zip` from your computer (`c:\Users\kitto\Desktop\hyper-local weather\colab_upload.zip`).

In [ ]:
import os, sys, shutil, zipfile

os.chdir("/content")
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# If ml_engine is not yet in /content, check for or prompt to upload colab_upload.zip
if not os.path.exists("/content/ml_engine"):
    zip_files = [f for f in os.listdir("/content") if f.endswith(".zip")]
    if not zip_files:
        print("📁 'colab_upload.zip' not found in this Colab session.")
        print("👉 Click the 'Choose Files' button below to select 'colab_upload.zip' from your computer:")
        from google.colab import files
        uploaded = files.upload()
        zip_files = [f for f in os.listdir("/content") if f.endswith(".zip")]

    if zip_files:
        target_zip = os.path.join("/content", zip_files[0])
        print(f"📦 Unpacking {target_zip}...")
        with zipfile.ZipFile(target_zip, 'r') as zf:
            for member in zf.infolist():
                member.filename = member.filename.replace("\\", "/")
                zf.extract(member, "/content")
        print("✅ Extraction complete!")

print("Folders in /content:", [d for d in os.listdir('/content') if os.path.isdir(os.path.join('/content', d))])

# Import and Execute the GPU Training Pipeline
if os.path.exists("/content/ml_engine"):
    from ml_engine.training_pipeline import run_training_and_benchmarking

    TARGET_REGION = "Nakuru_Njoro"
    MAX_RECORDS = 47450

    print(f"🚀 Starting GPU training for {TARGET_REGION} ({MAX_RECORDS} records)...")
    results = run_training_and_benchmarking(region_key=TARGET_REGION, max_records=MAX_RECORDS)
    print("\n🎉 Training and walk-forward benchmarking successfully completed!")
else:
    print("❌ Error: ml_engine folder not found. Please re-run this cell and upload 'colab_upload.zip'.")

### 4. Inspect Comparative Benchmark Evaluation Matrix (Objective v)

In [ ]:
import json
import pandas as pd
from ml_engine.config import WEIGHTS_DIR
from ml_engine.evaluation.metrics import format_metrics_table

report_json = WEIGHTS_DIR / "model_benchmark_report.json"
with open(report_json) as f:
    benchmark_data = json.load(f)

df_table = format_metrics_table(benchmark_data)
print("\n=== WALK-FORWARD VALIDATION BENCHMARK RESULTS ===")
display(df_table)

### 5. Visual Comparisons: R² Score, RMSE, and MAE across Architectures

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. R2 Score Comparison
sns.barplot(
    data=df_table, 
    x="Target Variable", 
    y="R² Score", 
    hue="Model Architecture", 
    ax=axes[0],
    palette="viridis"
)
axes[0].set_title("R² Score by Architecture (Higher is Better)", fontweight="bold")
axes[0].set_ylim(0, 1.0)
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# 2. RMSE Comparison
sns.barplot(
    data=df_table, 
    x="Target Variable", 
    y="RMSE", 
    hue="Model Architecture", 
    ax=axes[1],
    palette="rocket"
)
axes[1].set_title("Root Mean Squared Error (Lower is Better)", fontweight="bold")
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

### 6. Bundle and Download Production Weights to Your Computer
Run this cell to zip `ml_engine/weights/` and download `trained_weights.zip` directly in your browser.

In [ ]:
import shutil
from google.colab import files

# Create zip archive of weights directory
shutil.make_archive("trained_weights", "zip", str(WEIGHTS_DIR))
print("📦 Packaging complete: trained_weights.zip")
print("⬇️ Initiating download...")
files.download("trained_weights.zip")

### 7. Final Local Deployment Step
Once `trained_weights.zip` downloads:
1. Extract its contents directly into:
   `c:\Users\kitto\Desktop\hyper-local weather\ml_engine\weights\`
2. Files included:
   - `production_hybrid_lstm.pt`
   - `production_hybrid_xgb_*.json`
   - `production_hybrid_xgb_metadata.joblib`
   - `production_xgboost_*.json`
   - `production_xgboost_metadata.joblib`
   - `model_benchmark_report.json` and `.csv`
3. In your local terminal, start Django:
   ```bash
   python manage.py runserver
   ```
4. Visit `http://127.0.0.1:8000/` and `http://127.0.0.1:8000/admin-panel/ml/` to see your GPU-trained model in action!